# Transport regimes in CrSBr: from Arrhenius to canting to VRH

Single narrative notebook that combines four analyses already validated in the sibling notebooks `tmp/arrhenius_variable_Vprobe/`:

1. **Two-regime Arrhenius (c-axis)** — full $\ln|I|$ vs $1/T$ between 20 and 130 K at $V_{\rm probe}=0.20$ V, fitted as two straight lines: a low-T branch (20-80 K) and a high-T branch (90-130 K). Below 80 K the AFM and FM activation energies coincide; above 80 K they separate by $\sim 20$ meV.
2. **Full canting analysis (c-axis, high-T)** — one Arrhenius fit *per measured field* $H\parallel c$, giving a continuous $E_a(H)$. The shape is well described by $E_a^{\rm FM} + (E_a^{\rm AFM}-E_a^{\rm FM})\,(1-m^2)$ with $m=H/H_{\rm sat}^c$ — a quadratic / $(1-m^2)$ collapse onto the canting angle.
3. **Zabrodskii diagnostic below 80 K** — $W(T)=d\ln\sigma/d\ln T$ tested against Arrhenius / Mott / Efros-Shklovskii VRH. The fitted slope picks $p\approx 1/2$, i.e. ES-VRH-like transport, *not* thermal activation.
4. **Robustness across $V_{\rm probe}$** — repeat the per-$H$ Arrhenius fit and the $(1-m^2)$ canting fit for 10 probe biases in $[0.1, 0.4]$ V. The endpoint $E_a^{\rm AFM}$ and $E_a^{\rm FM}$ both drift with bias (consistent barrier tilt), but the $(1-m^2)$ shape is preserved and $\Delta E_a$ stays $\sim 16$ to $20$ meV across the full sweep.

All numerical values are reproduced from the validated sibling notebooks; nothing new is computed here that contradicts them.

**Operating point.** c-axis (hard axis, $H_{\rm sat}^c=1.45$ T). $V_{\rm probe}=0.20$ V with a $\pm 0.02$ V local linear window. Noise floor $|I|>1$ nA. AFM window $|H|<0.10$ T, FM window $|H|>H_{\rm sat}^c-0.10$ T.

In [ ]:
from scripts.utils import setup_notebook, OKABE_ITO_CYCLE
PROJECT_ROOT, np, pd, plt, Path = setup_notebook()
from scipy.optimize import curve_fit

## Configuration

Identical to the three sibling notebooks so the numbers here match `figures_two_regime/`, `figures_Ea_vs_H/`, and `figures_zabrodskii/` exactly. Outputs go to a per-run folder `tmp/arrhenius_variable_Vprobe/figures_story/`.

In [ ]:
V_PROBE          = 0.20     # V
V_PROBE_HW       = 0.02     # V    (half-window of local linear fit around V_PROBE)
I_FLOOR          = 1.0e-9   # A    (leakage / noise floor)
T_MIN            = 20       # K
T_LOW_MAX        = 80       # K    (upper end of low-T pinhole-like regime)
T_HIGH_MIN       = 90       # K    (lower end of high-T magnetically-sensitive regime)
T_MAX            = 130      # K    (upper cutoff, safely below T_N approx 132 K)
H_ROUND_DECIMALS = 2        #      merge floating-point H near-duplicates
N_T_MIN          = 4        #      min T points per H bin for a high-T Arrhenius fit

# c-axis (hard-axis canting) windows
H_SAT_C     = 1.45          # T    (saturation field, c-axis)
H_AFM_C_MAX = 0.10          # T
H_FM_C_MIN  = max(H_SAT_C - 0.10, 0.5 * H_SAT_C)

# Zabrodskii / VRH FM operating point (must be well above H_SAT_C)
H_FM_VRH    = 1.83          # T
H_FM_TOL    = 0.05          # T

k_B = 8.617333e-5           # eV/K

RAW_DIR_C = PROJECT_ROOT / 'output' / 'IV_H_scans' / 'dataframes' / 'c_scans'
OUT       = PROJECT_ROOT / 'tmp' / 'arrhenius_variable_Vprobe' / 'figures_story'
OUT.mkdir(parents=True, exist_ok=True)

ALL_T        = [3, 5, 7, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160]
TEMPERATURES = [T for T in ALL_T if T_MIN <= T <= T_MAX]
T_LOW        = [T for T in TEMPERATURES if T <= T_LOW_MAX]
T_HIGH       = [T for T in TEMPERATURES if T >= T_HIGH_MIN]

print(f'V_probe       : {V_PROBE} V (+/- {V_PROBE_HW} V window, |I| floor = {I_FLOOR*1e9:.2f} nA)')
print(f'Full T range  : [{T_MIN}, {T_MAX}] K')
print(f'Low-T regime  : T <= {T_LOW_MAX} K -> {T_LOW}')
print(f'High-T regime : T >= {T_HIGH_MIN} K -> {T_HIGH}')
print(f'Figures dir   : {OUT}')

## Shared helpers

Three building blocks reused by every step:

- `current_at_V` — per-curve local linear fit of $I(V)$ in $[V_{\rm probe}-h, V_{\rm probe}+h]$, evaluated at $V_{\rm probe}$. Returns $(H, I)$ for every IV curve in a per-$T$ pickle.
- `_linfit` — small weighted-or-unweighted linear fit returning slope, intercept, $\sigma_{\rm slope}$, $R^2$, $N$.
- `per_T_I_per_H` — group the IV curves at one temperature by rounded $H$ and report mean / std / $N$ per $(T, H)$ bin (above the noise floor).

In [ ]:
def current_at_V(df, V_probe=V_PROBE, half_window=V_PROBE_HW):
    """Per-curve I at fixed bias V_probe via a local linear fit of I vs V in
    [V_probe-hw, V_probe+hw], evaluated at V_probe.  Returns (H, I_at_V)."""
    Hs, Is = [], []
    for _, row in df.iterrows():
        v = np.asarray(row['voltage_smooth'])
        i = np.asarray(row['current_smooth'])
        m = (v >= V_probe - half_window) & (v <= V_probe + half_window)
        if m.sum() < 5:
            continue
        slope, intercept = np.polyfit(v[m], i[m], 1)
        I_at_V = slope * V_probe + intercept
        if np.isfinite(I_at_V):
            Hs.append(float(row['H']))
            Is.append(float(I_at_V))
    return np.array(Hs), np.array(Is)


def _linfit(x, y):
    x = np.asarray(x); y = np.asarray(y)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if x.size < 2:
        return np.nan, np.nan, np.nan, np.nan, 0
    coef, cov = np.polyfit(x, y, 1, cov=True)
    yhat = np.polyval(coef, x)
    ss_res = np.sum((y - yhat)**2); ss_tot = np.sum((y - y.mean())**2)
    R2 = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
    return float(coef[0]), float(coef[1]), float(np.sqrt(cov[0, 0])), float(R2), int(x.size)


def per_T_I_per_H(T, V_probe=V_PROBE, half_window=V_PROBE_HW):
    pkl = RAW_DIR_C / f'IV_gaussian_{T}K.pkl'
    df = pd.read_pickle(pkl)
    H_arr, I_arr = current_at_V(df, V_probe=V_probe, half_window=half_window)
    H_bin = np.round(H_arr, H_ROUND_DECIMALS)
    rows = []
    for h in np.unique(H_bin):
        in_bin = (H_bin == h)
        keep   = in_bin & (np.abs(I_arr) > I_FLOOR)
        if keep.sum() == 0:
            continue
        I_keep = I_arr[keep]
        rows.append(dict(
            T=T, H=float(h),
            I_mean=float(np.mean(I_keep)),
            I_std=float(np.std(I_keep, ddof=1)) if keep.sum() > 1 else np.nan,
            N=int(keep.sum())))
    return pd.DataFrame(rows)

## Step 1 — Two-regime Arrhenius (c-axis)

Endpoint currents at each temperature: average over the AFM field window ($|H|<0.10$ T) and the FM field window ($|H|>H_{\rm sat}^c-0.10$ T). Then fit $\ln|I|$ vs $1/T$ in two non-overlapping windows: $T\in[20,80]$ K and $T\in[90,130]$ K.

The full plot already shows curvature: a single Arrhenius line through all 12 temperatures cannot fit both ends, and the break sits at $\sim 85$ K. Splitting at the break gives two clean lines and a clear separation:

- **Low-T (20-80 K)**: $E_a^{\rm AFM} \approx E_a^{\rm FM} \approx 6.6$ meV. The activation energy is *the same* in the two magnetic states, i.e. the dominant low-T channel is magnetically blind. This is the regime that the Zabrodskii test in Step 3 will then identify as ES-VRH-like rather than truly activated.
- **High-T (90-130 K)**: $E_a^{\rm AFM} \approx 46.8$ meV, $E_a^{\rm FM} \approx 27.1$ meV. The endpoints now differ by $\Delta E_a \approx 19.7$ meV, in line with Lin et al. PRR **6**, 013185 (2024)'s band-edge $\Delta E_a^{\rm AFM-FM} \approx 20$ meV (memory: `lin_et_al_prr_2024`).

In [ ]:
def endpoints_at_T(T, V_probe=V_PROBE):
    pkl = RAW_DIR_C / f'IV_gaussian_{T}K.pkl'
    df = pd.read_pickle(pkl)
    H, I = current_at_V(df, V_probe=V_probe)
    aH = np.abs(H)
    H_FM_eff = min(H_FM_C_MIN, aH.max() - 0.05) if aH.size else np.nan
    afm = (aH < H_AFM_C_MAX) & (np.abs(I) > I_FLOOR)
    fm  = (aH > H_FM_eff)    & (np.abs(I) > I_FLOOR)
    return dict(T=T,
                n_afm=int(afm.sum()), n_fm=int(fm.sum()),
                I_afm_mean=float(np.mean(I[afm]))        if afm.any() else np.nan,
                I_afm_std =float(np.std(I[afm], ddof=1)) if afm.sum() > 1 else np.nan,
                I_fm_mean =float(np.mean(I[fm]))         if fm.any()  else np.nan,
                I_fm_std  =float(np.std(I[fm], ddof=1))  if fm.sum()  > 1 else np.nan)


ENDPT = {T: endpoints_at_T(T) for T in TEMPERATURES}
ENDPT_DF = pd.DataFrame([
    dict(T=T,
         lnI_afm=np.log(abs(s['I_afm_mean'])) if np.isfinite(s['I_afm_mean']) else np.nan,
         lnI_afm_err=(s['I_afm_std']/(abs(s['I_afm_mean'])*np.sqrt(s['n_afm'])))
             if (np.isfinite(s['I_afm_std']) and s['n_afm'] >= 2) else np.nan,
         lnI_fm=np.log(abs(s['I_fm_mean'])) if np.isfinite(s['I_fm_mean']) else np.nan,
         lnI_fm_err=(s['I_fm_std']/(abs(s['I_fm_mean'])*np.sqrt(s['n_fm'])))
             if (np.isfinite(s['I_fm_std']) and s['n_fm'] >= 2) else np.nan)
    for T, s in ENDPT.items()]).sort_values('T').reset_index(drop=True)

fit_low_afm  = _linfit(1.0/ENDPT_DF.loc[ENDPT_DF['T'] <= T_LOW_MAX, 'T'].values,
                       ENDPT_DF.loc[ENDPT_DF['T'] <= T_LOW_MAX, 'lnI_afm'].values)
fit_low_fm   = _linfit(1.0/ENDPT_DF.loc[ENDPT_DF['T'] <= T_LOW_MAX, 'T'].values,
                       ENDPT_DF.loc[ENDPT_DF['T'] <= T_LOW_MAX, 'lnI_fm'].values)
fit_high_afm = _linfit(1.0/ENDPT_DF.loc[ENDPT_DF['T'] >= T_HIGH_MIN, 'T'].values,
                       ENDPT_DF.loc[ENDPT_DF['T'] >= T_HIGH_MIN, 'lnI_afm'].values)
fit_high_fm  = _linfit(1.0/ENDPT_DF.loc[ENDPT_DF['T'] >= T_HIGH_MIN, 'T'].values,
                       ENDPT_DF.loc[ENDPT_DF['T'] >= T_HIGH_MIN, 'lnI_fm'].values)

REGIMES = {
    'low':  dict(afm=fit_low_afm,  fm=fit_low_fm,  T=T_LOW),
    'high': dict(afm=fit_high_afm, fm=fit_high_fm, T=T_HIGH),
}
for label, r in REGIMES.items():
    sA, _, sAe, R2A, nA = r['afm']
    sF, _, sFe, R2F, nF = r['fm']
    Ea_afm, Ea_afm_err = -sA*k_B*1000, sAe*k_B*1000
    Ea_fm,  Ea_fm_err  = -sF*k_B*1000, sFe*k_B*1000
    print(f'{label:4s} ({r["T"][0]} to {r["T"][-1]} K, N_AFM={nA}, N_FM={nF})')
    print(f'    E_a^AFM = {Ea_afm:6.2f} +/- {Ea_afm_err:5.2f} meV   R^2 = {R2A:.3f}')
    print(f'    E_a^FM  = {Ea_fm:6.2f} +/- {Ea_fm_err:5.2f} meV   R^2 = {R2F:.3f}')
    print(f'    dE (AFM - FM) = {Ea_afm - Ea_fm:+.2f} meV')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
c_afm, c_fm = OKABE_ITO_CYCLE[1], OKABE_ITO_CYCLE[5]
inv_T = 1.0 / ENDPT_DF['T'].values
in_low  = ENDPT_DF['T'].values <= T_LOW_MAX
in_high = ENDPT_DF['T'].values >= T_HIGH_MIN

ax.errorbar(inv_T[in_low], ENDPT_DF['lnI_afm'].values[in_low],
            yerr=ENDPT_DF['lnI_afm_err'].values[in_low],
            fmt='o', mec=c_afm, mfc=c_afm, ecolor=c_afm, ms=5, capsize=2,
            linestyle='none', label='AFM data')
ax.errorbar(inv_T[in_low], ENDPT_DF['lnI_fm'].values[in_low],
            yerr=ENDPT_DF['lnI_fm_err'].values[in_low],
            fmt='s', mec=c_fm, mfc=c_fm, ecolor=c_fm, ms=5, capsize=2,
            linestyle='none', label='FM data')
ax.errorbar(inv_T[in_high], ENDPT_DF['lnI_afm'].values[in_high],
            yerr=ENDPT_DF['lnI_afm_err'].values[in_high],
            fmt='o', mec=c_afm, mfc=c_afm, ecolor=c_afm, ms=5, capsize=2,
            linestyle='none')
ax.errorbar(inv_T[in_high], ENDPT_DF['lnI_fm'].values[in_high],
            yerr=ENDPT_DF['lnI_fm_err'].values[in_high],
            fmt='s', mec=c_fm, mfc=c_fm, ecolor=c_fm, ms=5, capsize=2,
            linestyle='none')

for regime, ls in [('low', '-'), ('high', '--')]:
    Tw = REGIMES[regime]['T']
    xg = np.linspace(1.0/Tw[-1]*0.97, 1.0/Tw[0]*1.03, 100)
    sA, bA, *_ = REGIMES[regime]['afm']
    sF, bF, *_ = REGIMES[regime]['fm']
    ax.plot(xg, sA*xg + bA, ls, color=c_afm, lw=1.2,
            label=f'AFM fit, {regime}-T')
    ax.plot(xg, sF*xg + bF, ls, color=c_fm,  lw=1.2,
            label=f'FM fit, {regime}-T')

ax.axvspan(1.0/T_HIGH_MIN, 1.0/T_LOW_MAX, color='0.88', alpha=0.5, zorder=0,
           label='Crossover gap')
ax.set_xlabel(r'$1/T$ (1/K)')
ax.set_ylabel(rf'$\ln|I (A)|$')
ax.legend(loc='lower left', framealpha=0.9)
fig.tight_layout()
fig.savefig(OUT / 'step1_two_regime_arrhenius_c_axis.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nLow-T regime (T <= 80 K): AFM and FM activation energies are equal within error')
print('    -> dominant low-T channel is magnetically blind (see Step 3 for hopping diagnosis)')
print('High-T regime (T >= 90 K): AFM is roughly 20 meV higher than FM')
print('    -> magnetically sensitive channel; Step 2 traces E_a continuously across the canting field')

In [ ]:
# ============================================================================
# Step 1b - Field-dependent two-regime Arrhenius (c-axis), with full propagation
# ============================================================================
# For each of 6 |H| targets spanning 0 to ~2 T along the c-axis (hard axis):
#   1. fold +H/-H bins (c-axis is symmetric -> doubles the statistics per cell)
#   2. inverse-variance-weighted combine the per-T mean currents across folded H bins
#   3. propagate sigma_I -> sigma_lnI via delta method: sigma_lnI = sigma_I / |I|
#   4. weighted linear fit of ln|I| vs 1/T in each regime
#         low-T   : T in [T_MIN, T_LOW_MAX]   = [20, 80] K
#         high-T  : T in [T_HIGH_MIN, T_MAX]  = [90, 130] K
#      using scipy curve_fit with absolute_sigma=True so sigma_lnI sets the
#      absolute error scale, then the parameter errors are inflated by
#      sqrt(chi2_red) whenever chi2_red > 1: sigma_lnI alone only captures
#      repeat-measurement precision, and a straight line does not fully
#      describe ln|I| vs 1/T over a 40-60 K window, so the raw covariance
#      understates the true slope uncertainty.
#   5. E_a = -k_B * slope,  sigma_Ea = k_B * sigma_slope (post-inflation)
#
# The figure shows data + errorbars connected with a dashed line (--o). The
# Arrhenius slopes (and their errors) are reported in the table below the figure
# and saved to CSV, but are NOT overlaid on the plot.

# Targets are snapped to actually measured fields on the 0.229 T grid:
H_TARGETS_C = np.array([0.000, 0.457, 0.914, 1.371, 1.829, 2.057])
H_MATCH_TOL = 0.05  # T


def linfit_weighted(x, y, sy):
    """Weighted linear fit of y = slope * x + intercept. Returns dict with
    parameters and their 1-sigma errors, R^2, chi^2, reduced chi^2, n.
    Errors start from the covariance at absolute_sigma=True (i.e. sigma_lnI
    sets the scale), then are inflated by sqrt(chi2_red) whenever chi2_red > 1
    (PDG scale-factor convention) so a returned slope_err also reflects real
    curvature/scatter about the line, not repeat-measurement precision alone."""
    x  = np.asarray(x,  float)
    y  = np.asarray(y,  float)
    sy = np.asarray(sy, float)
    m  = np.isfinite(x) & np.isfinite(y) & np.isfinite(sy) & (sy > 0)
    x, y, sy = x[m], y[m], sy[m]
    n = x.size
    if n < 2:
        return dict(slope=np.nan, intercept=np.nan, slope_err=np.nan,
                    intercept_err=np.nan, R2=np.nan, chi2=np.nan,
                    chi2_red=np.nan, n=n)
    popt, pcov = curve_fit(lambda xx, m_, b_: m_*xx + b_, x, y,
                           sigma=sy, absolute_sigma=True)
    slope, intercept = popt
    slope_err, intercept_err = np.sqrt(np.diag(pcov))
    yhat   = slope*x + intercept
    chi2   = float(np.sum(((y - yhat)/sy)**2))
    dof    = max(n - 2, 1)
    chi2_red = chi2/dof
    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - y.mean())**2)
    R2     = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
    scale  = np.sqrt(chi2_red) if chi2_red > 1 else 1.0
    slope_err     *= scale
    intercept_err *= scale
    return dict(slope=float(slope), intercept=float(intercept),
                slope_err=float(slope_err), intercept_err=float(intercept_err),
                R2=float(R2), chi2=chi2, chi2_red=chi2_red, n=n)


def collapse_field_at_T(T_subdf, H_target, tol):
    """At fixed T, fold +H/-H bins matching |H_target| within tol, then combine
    via inverse-variance weighting. Returns (I_comb, sigma_I_comb, N_total, n_bins).
    Falls back to a plain mean across bins if per-bin SEM is undefined (N=1)."""
    sel = T_subdf[np.abs(np.abs(T_subdf['H'].values) - H_target) <= tol]
    if len(sel) == 0:
        return np.nan, np.nan, 0, 0
    I, sd, N = sel['I_mean'].values, sel['I_std'].values, sel['N'].values
    keep     = np.abs(I) > I_FLOOR
    if not keep.any():
        return np.nan, np.nan, 0, 0
    I, sd, N = I[keep], sd[keep], N[keep]
    sem      = np.where((N >= 2) & np.isfinite(sd), sd/np.sqrt(N), np.nan)
    ok       = np.isfinite(sem) & (sem > 0)
    if ok.any():
        w = np.zeros_like(I, dtype=float)
        w[ok]    = 1.0/sem[ok]**2
        I_comb   = float(np.sum(w*I) / np.sum(w))
        sig_comb = float(1.0/np.sqrt(np.sum(w)))
    else:
        I_comb   = float(np.mean(I))
        sig_comb = float(np.std(I, ddof=1)/np.sqrt(len(I))) if len(I) > 1 else np.nan
    return I_comb, sig_comb, int(N.sum()), int(len(sel))


per_T_all = pd.concat([per_T_I_per_H(T) for T in TEMPERATURES], ignore_index=True)

# Build per-target ln|I| table with propagated errors, then weighted-fit each regime
traces_c, fits_c = {}, {}
for H_t in H_TARGETS_C:
    rows = []
    for T, g in per_T_all.groupby('T'):
        I, sI, Ntot, nbins = collapse_field_at_T(g, H_t, H_MATCH_TOL)
        if not np.isfinite(I) or abs(I) <= I_FLOOR:
            continue
        lnI     = np.log(abs(I))
        lnI_err = sI/abs(I) if np.isfinite(sI) else np.nan
        rows.append(dict(T=T, I=I, I_err=sI, lnI=lnI, lnI_err=lnI_err,
                         N_total=Ntot, n_bins=nbins))
    if not rows:
        traces_c[H_t] = pd.DataFrame(columns=['T','I','I_err','lnI','lnI_err','N_total','n_bins'])
        fits_c[H_t]   = dict(low=None, high=None)
        continue
    df = pd.DataFrame(rows).sort_values('T').reset_index(drop=True)
    traces_c[H_t] = df
    df_low, df_high = df[df['T'] <= T_LOW_MAX], df[df['T'] >= T_HIGH_MIN]
    fits_c[H_t] = dict(
        low =linfit_weighted(1.0/df_low ['T'].values, df_low ['lnI'].values, df_low ['lnI_err'].values) if len(df_low ) >= 2 else None,
        high=linfit_weighted(1.0/df_high['T'].values, df_high['lnI'].values, df_high['lnI_err'].values) if len(df_high) >= 2 else None)

# ---- Plot: data + errorbars + dashed cubic spline guide-to-the-eye per field
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
cmap = plt.cm.coolwarm
norm = plt.Normalize(vmin=H_TARGETS_C.min(), vmax=H_TARGETS_C.max())

for H_t in H_TARGETS_C:
    df = traces_c[H_t]
    if df.empty:
        continue
    c     = cmap(norm(H_t))
    inv_T = 1.0/df['T'].values
    order = np.argsort(inv_T)
    ax.errorbar(inv_T[order], df['lnI'].values[order],
                yerr=df['lnI_err'].values[order],
                fmt='--o', mec=c, mfc=c, color=c, ecolor=c,
                ms=5, capsize=2, lw=1.0, alpha=0.85)

ax.axvspan(1.0/T_HIGH_MIN, 1.0/T_LOW_MAX, color='0.88', alpha=0.5, zorder=0)
ax.set_xlabel(r'$1/T$ (1/K)')
ax.set_ylabel(rf'$\ln|I (A)|$')
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.85, pad=0.02)
cbar.set_label(r'$|H_{\mathrm{z}}|$ (T)')
fig.tight_layout()
fig.savefig(OUT / 'step1_field_dependent_arrhenius_c_axis.png', dpi=300, bbox_inches='tight')
plt.show()

# ---- Numerical table (cell output, also saved to CSV)
print('Per-field two-regime Arrhenius along the c-axis (folded +H / -H, weighted fit):\n')
print(f'{"|H|(T)":>7s}   {"Ea_low (meV)":>16s}  {"chi2r":>6s}  {"N":>3s}     {"Ea_high (meV)":>16s}  {"chi2r":>6s}  {"N":>3s}')
print('-'*92)
records = []
for H_t in H_TARGETS_C:
    f_low, f_high = fits_c[H_t]['low'], fits_c[H_t]['high']
    def _fmt(f):
        if f is None or not np.isfinite(f['slope']):
            return '         ---   ', np.nan, 0
        Ea     = -f['slope']*k_B*1000
        Ea_err =  f['slope_err']*k_B*1000
        return f'{Ea:6.2f} +/- {Ea_err:5.2f}', f['chi2_red'], f['n']
    low_str, chl, nl = _fmt(f_low)
    high_str, chh, nh = _fmt(f_high)
    print(f'{H_t:7.3f}   {low_str:>16s}  {chl:6.2f}  {nl:3d}     {high_str:>16s}  {chh:6.2f}  {nh:3d}')
    records.append(dict(
        H_T=H_t,
        Ea_low_meV     = -f_low ['slope']    *k_B*1000 if f_low  is not None else np.nan,
        Ea_low_err_meV =  f_low ['slope_err']*k_B*1000 if f_low  is not None else np.nan,
        chi2red_low    =  f_low ['chi2_red']           if f_low  is not None else np.nan,
        N_low          =  f_low ['n']                  if f_low  is not None else 0,
        Ea_high_meV    = -f_high['slope']    *k_B*1000 if f_high is not None else np.nan,
        Ea_high_err_meV=  f_high['slope_err']*k_B*1000 if f_high is not None else np.nan,
        chi2red_high   =  f_high['chi2_red']           if f_high is not None else np.nan,
        N_high         =  f_high['n']                  if f_high is not None else 0,
    ))
EA_FIELD_C = pd.DataFrame(records)
EA_FIELD_C.to_csv(OUT / 'step1_field_dependent_arrhenius_c_axis.csv', index=False)
print('\nSaved table to', OUT / 'step1_field_dependent_arrhenius_c_axis.csv')

In [ ]:
# ============================================================================
# Step 1b (V_probe = 0.80 V) - Field-dependent two-regime Arrhenius
# ============================================================================
# Same construction as Step 1b at 0.20 V, but evaluating the current at the
# higher probe bias V_probe = 0.80 V.  Reuses helpers (linfit_weighted,
# collapse_field_at_T, H_TARGETS_C, H_MATCH_TOL) already defined above.

V_08 = 0.80   # V

per_T_all_08 = pd.concat([per_T_I_per_H(T, V_probe=V_08) for T in TEMPERATURES],
                         ignore_index=True)

traces_c_08, fits_c_08 = {}, {}
for H_t in H_TARGETS_C:
    rows = []
    for T, g in per_T_all_08.groupby("T"):
        I, sI, Ntot, nbins = collapse_field_at_T(g, H_t, H_MATCH_TOL)
        if not np.isfinite(I) or abs(I) <= I_FLOOR:
            continue
        lnI     = np.log(abs(I))
        lnI_err = sI/abs(I) if np.isfinite(sI) else np.nan
        rows.append(dict(T=T, I=I, I_err=sI, lnI=lnI, lnI_err=lnI_err,
                         N_total=Ntot, n_bins=nbins))
    if not rows:
        traces_c_08[H_t] = pd.DataFrame(columns=["T","I","I_err","lnI","lnI_err","N_total","n_bins"])
        fits_c_08[H_t]   = dict(low=None, high=None)
        continue
    df = pd.DataFrame(rows).sort_values("T").reset_index(drop=True)
    traces_c_08[H_t] = df
    df_low  = df[df["T"] <= T_LOW_MAX]
    df_high = df[df["T"] >= T_HIGH_MIN]
    fits_c_08[H_t] = dict(
        low =linfit_weighted(1.0/df_low ["T"].values, df_low ["lnI"].values, df_low ["lnI_err"].values) if len(df_low ) >= 2 else None,
        high=linfit_weighted(1.0/df_high["T"].values, df_high["lnI"].values, df_high["lnI_err"].values) if len(df_high) >= 2 else None)

# ---- Plot
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
cmap08 = plt.cm.coolwarm
norm08 = plt.Normalize(vmin=H_TARGETS_C.min(), vmax=H_TARGETS_C.max())

for H_t in H_TARGETS_C:
    df = traces_c_08[H_t]
    if df.empty:
        continue
    c     = cmap08(norm08(H_t))
    inv_T = 1.0/df["T"].values
    order = np.argsort(inv_T)
    ax.errorbar(inv_T[order], df["lnI"].values[order],
                yerr=df["lnI_err"].values[order],
                fmt="--o", mec=c, mfc=c, color=c, ecolor=c,
                ms=5, capsize=2, lw=1.0, alpha=0.85)

ax.axvspan(1.0/T_HIGH_MIN, 1.0/T_LOW_MAX, color="0.88", alpha=0.5, zorder=0)
ax.set_xlabel(r"$1/T$ (1/K)")
ax.set_ylabel(r"$\ln|I\ (\mathrm{A})|$" + f"  at  $V_{{\mathrm{{probe}}}}={V_08:.2f}$ V")
sm08 = plt.cm.ScalarMappable(cmap=cmap08, norm=norm08); sm08.set_array([])
cbar08 = fig.colorbar(sm08, ax=ax, shrink=0.85, pad=0.02)
cbar08.set_label(r"$|H_{\mathrm{z}}|$ (T)")
fig.tight_layout()
fig.savefig(OUT / f"step1_field_dependent_arrhenius_c_axis_Vprobe_{V_08:.2f}V.png",
            dpi=300, bbox_inches="tight")
plt.show()

# ---- Numerical table
print(f"Per-field two-regime Arrhenius along the c-axis (V_probe = {V_08:.2f} V, folded +H/-H):\n")
print(f"{'|H|(T)':>7s}   {'Ea_low (meV)':>16s}  {'chi2r':>6s}  {'N':>3s}     {'Ea_high (meV)':>16s}  {'chi2r':>6s}  {'N':>3s}")
print("-"*92)
records_08 = []
for H_t in H_TARGETS_C:
    f_low, f_high = fits_c_08[H_t]["low"], fits_c_08[H_t]["high"]
    def _fmt(f):
        if f is None or not np.isfinite(f["slope"]):
            return "         ---   ", np.nan, 0
        Ea     = -f["slope"]*k_B*1000
        Ea_err =  f["slope_err"]*k_B*1000
        return f"{Ea:6.2f} +/- {Ea_err:5.2f}", f["chi2_red"], f["n"]
    low_str,  chl, nl = _fmt(f_low)
    high_str, chh, nh = _fmt(f_high)
    print(f"{H_t:7.3f}   {low_str:>16s}  {chl:6.2f}  {nl:3d}     {high_str:>16s}  {chh:6.2f}  {nh:3d}")
    records_08.append(dict(
        H_T=H_t,
        Ea_low_meV     = -f_low ["slope"]    *k_B*1000 if f_low  is not None else np.nan,
        Ea_low_err_meV =  f_low ["slope_err"]*k_B*1000 if f_low  is not None else np.nan,
        chi2red_low    =  f_low ["chi2_red"]            if f_low  is not None else np.nan,
        N_low          =  f_low ["n"]                   if f_low  is not None else 0,
        Ea_high_meV    = -f_high["slope"]    *k_B*1000 if f_high is not None else np.nan,
        Ea_high_err_meV=  f_high["slope_err"]*k_B*1000 if f_high is not None else np.nan,
        chi2red_high   =  f_high["chi2_red"]            if f_high is not None else np.nan,
        N_high         =  f_high["n"]                   if f_high is not None else 0,
    ))
EA_FIELD_C_08 = pd.DataFrame(records_08)
EA_FIELD_C_08.to_csv(OUT / f"step1_field_dependent_arrhenius_c_axis_Vprobe_{V_08:.2f}V.csv", index=False)
print("\nSaved table to", OUT / f"step1_field_dependent_arrhenius_c_axis_Vprobe_{V_08:.2f}V.csv")


In [ ]:
# ============================================================================
# Step 1c - ES-VRH form: ln|I| vs T^(-1/2) at the same 6 |H| fields
# ============================================================================
# Efros-Shklovskii VRH:  sigma propto exp[-(T_0/T)^(1/2)]
#                ->  ln|I| = const - (T_0)^(1/2) * T^(-1/2)
# Straight lines on a ln|I| vs T^(-1/2) plot indicate ES-VRH-like transport.
# Weighted linear fit on the low-T branch (T <= T_LOW_MAX) gives T_0 = slope^2,
# with sigma_T0 = 2 |slope| sigma_slope (delta method).
# Reuses traces_c, cmap, norm, linfit_weighted from the previous cell.

fits_es = {}
for H_t in H_TARGETS_C:
    df = traces_c[H_t]
    if df.empty:
        fits_es[H_t] = None
        continue
    df_low = df[df['T'] <= T_LOW_MAX]
    if len(df_low) < 2:
        fits_es[H_t] = None
        continue
    x_es = 1.0/np.sqrt(df_low['T'].values)
    fits_es[H_t] = linfit_weighted(x_es, df_low['lnI'].values, df_low['lnI_err'].values)

# ---- Plot: data + errorbars with dashed line connector (--o), no separate spline
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
for H_t in H_TARGETS_C:
    df = traces_c[H_t]
    if df.empty:
        continue
    c     = cmap(norm(H_t))
    x_es  = 1.0/np.sqrt(df['T'].values)
    order = np.argsort(x_es)
    ax.errorbar(x_es[order], df['lnI'].values[order],
                yerr=df['lnI_err'].values[order],
                fmt='--o', mec=c, mfc=c, color=c, ecolor=c,
                ms=5, capsize=2, lw=1.0, alpha=0.85)

ax.axvspan(1.0/np.sqrt(T_HIGH_MIN), 1.0/np.sqrt(T_LOW_MAX),
           color='0.88', alpha=0.5, zorder=0)
ax.set_xlabel(r'$T^{-1/2}$ (K$^{-1/2}$)')
ax.set_ylabel(rf'$\ln|I (A)|$')
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.85, pad=0.02)
cbar.set_label(r'$|H_\mathrm{z}|$ (T)')
fig.tight_layout()
fig.savefig(OUT / 'step1_field_dependent_ES_VRH_c_axis.png', dpi=300, bbox_inches='tight')
plt.show()

# ---- Numerical table: low-T ES-VRH fits per field
print('Per-field low-T ES-VRH fits along the c-axis (ln|I| vs T^(-1/2), T <= '
      f'{T_LOW_MAX} K, folded +H / -H, weighted):\n')
print(f'{"|H|(T)":>7s}   {"slope (K^1/2)":>17s}   {"T_0 (K)":>14s}   {"chi2r":>6s}  {"N":>3s}')
print('-'*72)
es_rows = []
for H_t in H_TARGETS_C:
    f = fits_es[H_t]
    if f is None or not np.isfinite(f['slope']):
        print(f'{H_t:7.3f}                  ---                 ---       ---    --')
        es_rows.append(dict(H_T=H_t, slope=np.nan, slope_err=np.nan,
                            T0_K=np.nan, T0_err_K=np.nan, chi2red=np.nan, N=0))
        continue
    s, se   = f['slope'], f['slope_err']
    T0      = s**2
    T0_err  = 2.0*abs(s)*se
    print(f'{H_t:7.3f}   {s:7.2f} +/- {se:5.2f}    {T0:7.1f} +/- {T0_err:5.1f}   {f["chi2_red"]:6.2f}  {f["n"]:3d}')
    es_rows.append(dict(H_T=H_t, slope=s, slope_err=se,
                        T0_K=T0, T0_err_K=T0_err,
                        chi2red=f['chi2_red'], N=f['n']))
ES_FIELD_C = pd.DataFrame(es_rows)
ES_FIELD_C.to_csv(OUT / 'step1_field_dependent_ES_VRH_c_axis.csv', index=False)
print('\nSaved table to', OUT / 'step1_field_dependent_ES_VRH_c_axis.csv')

In [ ]:
# ============================================================================
# Step 1d - Low-T (T <= 80 K) Arrhenius comparison at two V_probe biases
# ============================================================================
# Same field-dependent ln|I| vs 1/T construction as Step 1b, restricted to the
# low-T branch only, computed at V_probe = 0.20 V and V_probe = 0.80 V.
# Same 6 |H| targets, same +/-H folding, same inverse-variance combine, same
# delta-method error propagation. Reuses H_TARGETS_C, H_MATCH_TOL,
# collapse_field_at_T, cmap, norm from previous cells.

V_PANELS = [0.20, 0.80]


def build_low_T_traces(V_probe):
    per_T_low_local = pd.concat(
        [per_T_I_per_H(T, V_probe=V_probe) for T in T_LOW],
        ignore_index=True)
    out = {}
    for H_t in H_TARGETS_C:
        rows = []
        for T, g in per_T_low_local.groupby('T'):
            I, sI, Ntot, nbins = collapse_field_at_T(g, H_t, H_MATCH_TOL)
            if not np.isfinite(I) or abs(I) <= I_FLOOR:
                continue
            lnI     = np.log(abs(I))
            lnI_err = sI/abs(I) if np.isfinite(sI) else np.nan
            rows.append(dict(T=T, lnI=lnI, lnI_err=lnI_err))
        out[H_t] = (pd.DataFrame(rows).sort_values('T').reset_index(drop=True)
                    if rows else pd.DataFrame(columns=['T','lnI','lnI_err']))
    return out


fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=300)

for ax, V in zip(axes, V_PANELS):
    traces_V = build_low_T_traces(V)
    for H_t in H_TARGETS_C:
        df = traces_V[H_t]
        if df.empty:
            continue
        c     = cmap(norm(H_t))
        inv_T = 1.0/df['T'].values
        order = np.argsort(inv_T)
        ax.errorbar(inv_T[order], df['lnI'].values[order],
                    yerr=df['lnI_err'].values[order],
                    fmt='--o', mec=c, mfc=c, color=c, ecolor=c,
                    ms=5, capsize=2, lw=1.0, alpha=0.85)
    ax.set_xlabel(r'$1/T$ (1/K)')
    ax.text(0.04, 0.05, rf'$V_\mathrm{{probe}} = {V:.2f}$ V',
            transform=ax.transAxes, ha='left', va='bottom')

axes[0].set_ylabel(r'$\ln|I\,(\mathrm{A})|$')
axes[1].set_ylabel(r'$\ln|I\,(\mathrm{A})|$')

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, shrink=0.85, pad=0.02)
cbar.set_label(r'$|H_\mathrm{z}|$ (T)')

fig.savefig(OUT / 'step1_lowT_arrhenius_two_Vprobe.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# Step 1e - High-T (T >= 90 K) Arrhenius, c-axis, single V_probe panel
#           UNWEIGHTED linear fit; visible data error bars.
# ============================================================================
K_B_eV = 8.617333262e-5   # eV/K

norm_local = plt.Normalize(vmin=H_TARGETS_C.min(), vmax=H_TARGETS_C.max())

fig, ax = plt.subplots(figsize=(6, 5), dpi=300)

fit_summary = []
for H_t in H_TARGETS_C:
    df = traces_c[H_t]
    if df.empty:
        continue
    df_high = df[df['T'] >= T_HIGH_MIN]
    if len(df_high) < 2:
        continue
    c     = cmap(norm_local(H_t))
    inv_T = 1.0/df_high['T'].values
    lnI   = df_high['lnI'].values
    yerr  = df_high['lnI_err'].values
    # Replace non-finite errors with zero so the marker is still drawn
    yerr_plot = np.where(np.isfinite(yerr), yerr, 0.0)
    order = np.argsort(inv_T)

    ax.errorbar(inv_T[order], lnI[order], yerr=yerr_plot[order],
                fmt='o', mec=c, mfc=c, color=c, ecolor=c,
                ms=5, capsize=4, elinewidth=1.2, capthick=1.2,
                lw=0, alpha=0.9, zorder=3)

    # Unweighted linear fit
    coef, cov = np.polyfit(inv_T, lnI, 1, cov=True)
    slope, intercept = coef
    slope_se, intercept_se = np.sqrt(np.diag(cov))

    xfit = np.linspace(inv_T.min(), inv_T.max(), 100)
    yfit = intercept + slope*xfit
    # 1-sigma confidence band on the fit line: var(y) = sa^2 + x^2*sb^2 + 2x*cov
    y_var = (intercept_se**2 + (xfit**2)*(slope_se**2)
             + 2*xfit*cov[0, 1])
    y_sig = np.sqrt(np.clip(y_var, 0, None))
    ax.plot(xfit, yfit, '-', color=c, lw=1.2, alpha=0.9, zorder=2)
    ax.fill_between(xfit, yfit - y_sig, yfit + y_sig,
                    color=c, alpha=0.18, lw=0, zorder=1)

    Ea_meV    = -slope    * K_B_eV * 1e3
    Ea_meV_se =  slope_se * K_B_eV * 1e3
    fit_summary.append(dict(H_T=H_t, slope=slope, slope_se=slope_se,
                            Ea_meV=Ea_meV, Ea_meV_se=Ea_meV_se,
                            n=len(df_high)))

ax.set_xlabel(r'$1/T$ (1/K)')
ax.set_ylabel(r'$\ln|I\,(\mathrm{A})|$')
ax.text(0.04, 0.05,
        rf'c-axis,  $V_\mathrm{{probe}} = {V_PROBE:.2f}$ V',
        transform=ax.transAxes, ha='left', va='bottom')

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm_local); sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.85, pad=0.02)
cbar.set_label(r'$|H_\mathrm{z}|$ (T)')

fig.savefig(OUT / f'step1_highT_arrhenius_c_axis_Vprobe_{V_PROBE:.2f}V_unweighted.png',
            dpi=300, bbox_inches='tight')
plt.show()

print(f'High-T Arrhenius fits along the c-axis (unweighted, '
      f'T in [{T_HIGH_MIN}, {max(T_HIGH)}] K, '
      f'V_probe = {V_PROBE:.2f} V):\n')
print(f"{'|H| (T)':>8}  {'slope (K)':>20}  {'E_a (meV)':>20}  {'n':>3}")
for r in fit_summary:
    print(f"{r['H_T']:8.2f}  {r['slope']:10.1f} +/- {r['slope_se']:6.1f}"
          f"   {r['Ea_meV']:9.2f} +/- {r['Ea_meV_se']:6.2f}"
          f"   {r['n']:3d}")


## Step 2 — Canting analysis: $E_a(H)$ along the c-axis

Drop the AFM / FM window averaging. At each measured field $H$ along the c-axis, do a stand-alone Arrhenius fit on the five high-T points (90, 100, 110, 120, 130 K) to get $E_a(H)$. The c-axis is the hard axis, so between $H=0$ and $H_{\rm sat}^c=1.45$ T the sublattice moments cant continuously — every $H$ corresponds to a definite canting angle.

**Expected shape.** If the high-T activation energy is set by the AFM/FM band-edge shift (Lin et al. PRR 2024 picture), it should follow
$$E_a(H) \;=\; E_a^{\rm FM} + (E_a^{\rm AFM}-E_a^{\rm FM})\,(1-m^2), \qquad m = \mathrm{clip}(|H|/H_{\rm sat}^c, 0, 1),$$
i.e. parabolic in $H$ in the canting region with curvature $b = -(E_a^{\rm AFM}-E_a^{\rm FM})/(H_{\rm sat}^c)^2$, flattening to $E_a^{\rm FM}$ once the system saturates. This is the $(1-m^2)$ collapse and it is exactly what the data show below.

In [ ]:
def Ea_vs_H(V_probe=V_PROBE):
    per_T = [per_T_I_per_H(T, V_probe=V_probe) for T in T_HIGH]
    IH = pd.concat(per_T, ignore_index=True)
    rows = []
    for h_bin, sub in IH.groupby('H'):
        sub = sub.sort_values('T')
        if len(sub) < N_T_MIN:
            continue
        slope, intercept, slope_err, R2, n = _linfit(1.0/sub['T'].values,
                                                     np.log(np.abs(sub['I_mean'].values)))
        rows.append(dict(H=h_bin, N_T=n,
                         Ea_meV=-slope*k_B*1000,
                         Ea_err_meV=slope_err*k_B*1000, R2=R2,
                         slope=slope, intercept=intercept))
    return IH, pd.DataFrame(rows).sort_values('H').reset_index(drop=True)


def cant_model(H, E_AFM, E_FM, H_sat=H_SAT_C):
    m = np.clip(np.abs(H) / H_sat, 0.0, 1.0)
    return E_FM + (E_AFM - E_FM) * (1.0 - m**2)


IH_main, EA_main = Ea_vs_H(V_PROBE)
popt, pcov = curve_fit(cant_model, EA_main['H'].values, EA_main['Ea_meV'].values,
                       sigma=EA_main['Ea_err_meV'].values, absolute_sigma=True,
                       p0=(46.8, 27.1), maxfev=5000)
E_AFM_REF, E_FM_REF = popt
E_AFM_REF_err, E_FM_REF_err = np.sqrt(np.diag(pcov))
print(f'Cant-model fit at V_probe = {V_PROBE} V:')
print(f'    E_a^AFM = {E_AFM_REF:.2f} +/- {E_AFM_REF_err:.2f} meV')
print(f'    E_a^FM  = {E_FM_REF:.2f} +/- {E_FM_REF_err:.2f} meV')
print(f'    Delta E_a = {E_AFM_REF - E_FM_REF:.2f} meV')
print(f'    curvature b = -(dE)/H_sat^2 = {-(E_AFM_REF - E_FM_REF)/H_SAT_C**2:.2f} meV/T^2')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
ax.axvspan(-H_SAT_C, H_SAT_C, color='0.88', alpha=0.4, zorder=0,
           label=f'$|H|<H_{{\\rm sat}}^c={H_SAT_C:.2f}$ T')
ax.errorbar(EA_main['H'], EA_main['Ea_meV'], yerr=EA_main['Ea_err_meV'],
            fmt='o', color=OKABE_ITO_CYCLE[0], ms=5, capsize=2, lw=1.0,
            label='Data')
H_grid = np.linspace(EA_main['H'].min(), EA_main['H'].max(), 400)
ax.plot(H_grid, cant_model(H_grid, E_AFM_REF, E_FM_REF), '--',
        color='0.3', lw=1.2, label=r'$(1-m^2)$ fit')
ax.axhline(0, color='0.5', lw=0.5)
ax.axvline(0, color='0.5', lw=0.5)
ax.set_xlabel(r'$H_{\mathrm{z}}$ (T)')
ax.set_ylabel(r'$E_a$ (meV)')
ax.set_ylim(bottom=15)
ax.legend(loc='lower center', framealpha=0.9, fontsize=15)
fig.tight_layout()
fig.savefig(OUT / 'step2_Ea_vs_H_cant_fit.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 3 — Zabrodskii test below 80 K: not Arrhenius, but ES-VRH-like

Step 1 fit the low-T branch with Arrhenius and got $E_a \approx 6.6$ meV. Step 1 *did not* prove that the underlying transport is thermal activation — many functional forms look approximately linear in $\ln|I|$ vs $1/T$ over a single half-decade. The Zabrodskii diagnostic
$$ W(T) \equiv \frac{d\ln\sigma}{d\ln T}, \qquad \sigma \propto \exp[-(T_0/T)^p] \;\Rightarrow\; \ln W = -p\ln T + \text{const} $$
turns the question into a slope measurement: $p=1$ is Arrhenius, $p=1/2$ is Efros-Shklovskii VRH, $p=1/4$ is 3D Mott VRH.

Two operating points: AFM ($H=0$) and FM ($|H|=1.83$ T, well above $H_{\rm sat}^c$, $+H/-H$ folded). Six T-points (20-70 K). Result, validated in the sibling notebook:

- AFM slope $= -0.57 \pm 0.08$ ($p\approx 0.57$). Arrhenius is $5.4\sigma$ away; ES VRH is within $0.9\sigma$.
- FM slope $= -0.48 \pm 0.03$ ($p\approx 0.48$). Arrhenius is $15\sigma$ away; ES VRH is within $0.6\sigma$.

Both AFM and FM pick the same exponent ($\sim 1/2$), and the $6.6$ meV "$E_a$" from Step 1 should be reinterpreted as the apparent activation energy of an ES-VRH-like channel evaluated over this $T$ window — not as a real barrier height. This is also why Step 1 found $E_a^{\rm AFM}=E_a^{\rm FM}$: the low-T transport is dominated by the magnetically blind hopping channel.

In [ ]:
def collapse_to_state(IH, H_target, tol, fold_pm=True):
    out = []
    for T, g in IH.groupby('T'):
        if abs(H_target) < tol:
            sel = g[np.isclose(g['H'], 0.0, atol=tol)]
        else:
            sel = g[(np.abs(np.abs(g['H']) - H_target) < tol)] if fold_pm \
                  else g[np.isclose(g['H'], H_target, atol=tol)]
        if len(sel) == 0:
            continue
        I  = sel['I_mean'].values
        sd = sel['I_std'].values
        N  = sel['N'].values
        sem = sd / np.sqrt(N)
        w   = np.where(np.isfinite(sem) & (sem > 0), 1.0/sem**2, np.nan)
        if np.all(~np.isfinite(w)):
            I_w   = float(np.mean(I))
            sem_w = float(np.std(I, ddof=1)/np.sqrt(len(I))) if len(I) > 1 else np.nan
        else:
            ww = np.where(np.isfinite(w), w, 0.0)
            I_w   = float(np.sum(ww*I) / np.sum(ww))
            sem_w = float(1.0/np.sqrt(np.sum(ww)))
        out.append(dict(T=T, I=I_w, sem=sem_w, N_bins=len(sel)))
    return pd.DataFrame(out).sort_values('T').reset_index(drop=True)


def log_log_derivative(T, I, sem):
    T = np.asarray(T, float); I = np.asarray(I, float); sem = np.asarray(sem, float)
    lnT = np.log(T); lnI = np.log(np.abs(I)); sig_lnI = sem / np.abs(I)
    n = len(T)
    W     = np.full(n, np.nan)
    sig_W = np.full(n, np.nan)
    for i in range(1, n-1):
        dlnT = lnT[i+1] - lnT[i-1]
        W[i]     = (lnI[i+1] - lnI[i-1]) / dlnT
        sig_W[i] = np.sqrt(sig_lnI[i+1]**2 + sig_lnI[i-1]**2) / abs(dlnT)
    dlnT0 = lnT[1] - lnT[0]
    W[0]     = (lnI[1] - lnI[0]) / dlnT0
    sig_W[0] = np.sqrt(sig_lnI[1]**2 + sig_lnI[0]**2) / abs(dlnT0)
    dlnTn = lnT[-1] - lnT[-2]
    W[-1]     = (lnI[-1] - lnI[-2]) / dlnTn
    sig_W[-1] = np.sqrt(sig_lnI[-1]**2 + sig_lnI[-2]**2) / abs(dlnTn)
    T_eff = np.empty(n)
    T_eff[0]    = np.sqrt(T[0]*T[1])
    T_eff[-1]   = np.sqrt(T[-1]*T[-2])
    T_eff[1:-1] = np.sqrt(T[:-2]*T[2:])
    return T_eff, W, sig_W


def zab_slope(T_eff, W, sig_W):
    m = np.isfinite(W) & (W > 0)
    x = np.log(T_eff[m]); y = np.log(W[m]); sy = sig_W[m] / W[m]
    w = 1.0 / np.maximum(sy, 1e-12)
    coef, cov = np.polyfit(x, y, 1, w=w, cov=True)
    yhat = np.polyval(coef, x)
    ss_res = np.sum(((y - yhat) * w)**2)
    ss_tot = np.sum(((y - np.average(y, weights=w**2)) * w)**2)
    R2 = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
    return dict(slope=float(coef[0]), p=float(-coef[0]),
                intercept=float(coef[1]),
                slope_err=float(np.sqrt(cov[0, 0])), R2=float(R2), n=int(x.size))


per_T_low = [per_T_I_per_H(T) for T in T_LOW]
IH_low    = pd.concat(per_T_low, ignore_index=True)
AFM_IT    = collapse_to_state(IH_low, 0.0,        tol=10**(-H_ROUND_DECIMALS)*1.5, fold_pm=False)
FM_IT     = collapse_to_state(IH_low, H_FM_VRH,   tol=H_FM_TOL,                    fold_pm=True)

T_e_A, W_A, sW_A = log_log_derivative(AFM_IT['T'], AFM_IT['I'], AFM_IT['sem'])
T_e_F, W_F, sW_F = log_log_derivative(FM_IT ['T'], FM_IT ['I'], FM_IT ['sem'])
fit_A = zab_slope(T_e_A, W_A, sW_A)
fit_F = zab_slope(T_e_F, W_F, sW_F)

MODELS = {'Arrhenius': 1.00, 'ES VRH': 0.50, '2D Mott VRH': 1.0/3.0, '3D Mott VRH': 0.25}
print('Zabrodskii slopes (c-axis, T in [20, 70] K, V_probe = 0.20 V):')
for label, fit in [('AFM (H = 0)', fit_A), (f'FM  (|H| = {H_FM_VRH:.2f} T)', fit_F)]:
    print(f'  {label}:  slope = {fit["slope"]:+.3f} +/- {fit["slope_err"]:.3f}   '
          f'(p = {fit["p"]:+.3f},  R^2 = {fit["R2"]:.3f},  n = {fit["n"]})')
print('\nDeviation from candidate models (sigma on slope):')
for name, p_model in MODELS.items():
    sA = (fit_A['slope'] - (-p_model)) / fit_A['slope_err']
    sF = (fit_F['slope'] - (-p_model)) / fit_F['slope_err']
    print(f'  {name:14s} (p={p_model:.2f}):   AFM = {sA:+5.2f} sigma   FM = {sF:+5.2f} sigma')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
AFM_COLOR = OKABE_ITO_CYCLE[1]; FM_COLOR = OKABE_ITO_CYCLE[5]
ax.errorbar(T_e_A, W_A, yerr=sW_A, fmt='o', ms=5, capsize=2, lw=1.0, color=AFM_COLOR,
            label=f'AFM ($H=0$),  slope $= {fit_A["slope"]:+.2f}\\pm{fit_A["slope_err"]:.2f}$')
ax.errorbar(T_e_F, W_F, yerr=sW_F, fmt='s', ms=5, capsize=2, lw=1.0, color=FM_COLOR,
            label=f'FM ($|H|={H_FM_VRH:.2f}$ T),  slope $= {fit_F["slope"]:+.2f}\\pm{fit_F["slope_err"]:.2f}$')
T_line = np.geomspace(min(T_e_A.min(), T_e_F.min())*0.9,
                       max(T_e_A.max(), T_e_F.max())*1.1, 100)
for fit, color in [(fit_A, AFM_COLOR), (fit_F, FM_COLOR)]:
    ax.plot(T_line, np.exp(fit['intercept']) * T_line**fit['slope'], '-',
            color=color, lw=1.2, alpha=0.8)
# Reference power-law slopes through AFM centroid
T_c = np.exp(np.mean(np.log(T_e_A[np.isfinite(W_A) & (W_A > 0)])))
W_c = np.exp(np.mean(np.log(W_A [np.isfinite(W_A) & (W_A > 0)])))
for (name, p_model), c in zip(MODELS.items(), ['0.30', '0.45', '0.60', '0.75']):
    ax.plot(T_line, W_c * (T_line/T_c)**(-p_model), '--', color=c, lw=0.9,
            label=f'{name}, slope = ${-p_model:+.2f}$')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'$T_{\rm eff}$ (K)')
ax.set_ylabel(r'$W = d\,\ln|I|\,/\,d\,\ln T$')
ax.legend(loc='lower left', framealpha=0.9)
ax.grid(True, which='both', alpha=0.25)
fig.tight_layout()
fig.savefig(OUT / 'step3_zabrodskii_lowT.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 4 — Robustness: sweep $V_{\rm probe}$ and refit

If the picture is right, two things should be true across $V_{\rm probe}$:

1. The per-$H$ $E_a(H)$ curve keeps the same $(1-m^2)$ shape — i.e. the **shape** of the canting response is bias-independent.
2. Both endpoints $E_a^{\rm AFM}$ and $E_a^{\rm FM}$ may drift with bias (barrier tilt), but their **difference** $\Delta E_a$ stays in the same $\sim 16$ to $20$ meV band.

Sweep $V_{\rm probe}\in[0.10, 0.40]$ V at 10 evenly spaced biases; refit per-$H$ Arrhenius and refit the cant model. The two cells below are exactly the plots in the user's attached figures.

In [ ]:
V_PROBE_SWEEP = np.linspace(0.10, 0.40, 10)
EA_sweep_list = []
fit_rows = []
for V in V_PROBE_SWEEP:
    _, ea = Ea_vs_H(V_probe=V)
    EA_sweep_list.append(ea)
    popt, pcov = curve_fit(cant_model, ea['H'].values, ea['Ea_meV'].values,
                           sigma=ea['Ea_err_meV'].values, absolute_sigma=True,
                           p0=(E_AFM_REF, E_FM_REF), maxfev=5000)
    EA_fit, EF_fit = popt
    EA_err, EF_err = np.sqrt(np.diag(pcov))
    b      = -(EA_fit - EF_fit) / H_SAT_C**2
    var_b  = (pcov[0, 0] + pcov[1, 1] - 2.0 * pcov[0, 1]) / H_SAT_C**4
    b_err  = float(np.sqrt(max(var_b, 0.0)))
    pred   = cant_model(ea['H'].values, *popt)
    chi2r  = float(np.sum(((ea['Ea_meV'].values - pred) / ea['Ea_err_meV'].values)**2)
                   / max(len(ea) - 2, 1))
    fit_rows.append(dict(V_probe=V,
                         E_AFM_meV=EA_fit, E_AFM_err_meV=EA_err,
                         E_FM_meV=EF_fit,  E_FM_err_meV=EF_err,
                         dE_meV=EA_fit - EF_fit,
                         curvature_b=b, curvature_b_err=b_err,
                         chi2_reduced=chi2r, N=len(ea)))
EA_sweep = pd.concat([ea.assign(V_probe=V) for V, ea in zip(V_PROBE_SWEEP, EA_sweep_list)],
                    ignore_index=True)
FITS = pd.DataFrame(fit_rows)
print(FITS.round(3).to_string(index=False))

In [ ]:
#============================================================================
# Step 1b - Field-dependent two-regime Arrhenius (c-axis), with full propagation
# ============================================================================
# For each of 6 |H| targets spanning 0 to ~2 T along the c-axis (hard axis):
#   1. fold +H/-H bins (c-axis is symmetric -> doubles the statistics per cell)
#   2. inverse-variance-weighted combine the per-T mean currents across folded H bins
#   3. propagate sigma_I -> sigma_lnI via delta method: sigma_lnI = sigma_I / |I|
#   4. weighted linear fit of ln|I| vs 1/T in each regime
#         low-T   : T in [T_MIN, T_LOW_MAX]   = [20, 80] K
#         high-T  : T in [T_HIGH_MIN, T_MAX]  = [90, 130] K
#      using scipy curve_fit with absolute_sigma=True so the parameter errors
#      come straight from sigma_lnI (not rescaled by reduced chi^2).
#   5. E_a = -k_B * slope,  sigma_Ea = k_B * sigma_slope
#
# The figure shows data + errorbars connected with a dashed line (--o). The
# Arrhenius slopes (and their errors) are reported in the table below the figure
# and saved to CSV, but are NOT overlaid on the plot.

# Targets are snapped to actually measured fields on the 0.229 T grid:
H_TARGETS_C = np.array([0.000, 0.457, 0.914, 1.371, 1.829, 2.057])
H_MATCH_TOL = 0.05  # T


def linfit_weighted(x, y, sy):
    """Weighted linear fit of y = slope * x + intercept. Returns dict with
    parameters, their 1-sigma errors (from the covariance, absolute_sigma=True),
    R^2, chi^2, reduced chi^2, n."""
    x  = np.asarray(x,  float)
    y  = np.asarray(y,  float)
    sy = np.asarray(sy, float)
    m  = np.isfinite(x) & np.isfinite(y) & np.isfinite(sy) & (sy > 0)
    x, y, sy = x[m], y[m], sy[m]
    n = x.size
    if n < 2:
        return dict(slope=np.nan, intercept=np.nan, slope_err=np.nan,
                    intercept_err=np.nan, R2=np.nan, chi2=np.nan,
                    chi2_red=np.nan, n=n)
    popt, pcov = curve_fit(lambda xx, m_, b_: m_*xx + b_, x, y,
                           sigma=sy, absolute_sigma=True)
    slope, intercept = popt
    slope_err, intercept_err = np.sqrt(np.diag(pcov))
    yhat   = slope*x + intercept
    chi2   = float(np.sum(((y - yhat)/sy)**2))
    dof    = max(n - 2, 1)
    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - y.mean())**2)
    R2     = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
    return dict(slope=float(slope), intercept=float(intercept),
                slope_err=float(slope_err), intercept_err=float(intercept_err),
                R2=float(R2), chi2=chi2, chi2_red=chi2/dof, n=n)


def collapse_field_at_T(T_subdf, H_target, tol):
    """At fixed T, fold +H/-H bins matching |H_target| within tol, then combine
    via inverse-variance weighting. Returns (I_comb, sigma_I_comb, N_total, n_bins).
    Falls back to a plain mean across bins if per-bin SEM is undefined (N=1)."""
    sel = T_subdf[np.abs(np.abs(T_subdf['H'].values) - H_target) <= tol]
    if len(sel) == 0:
        return np.nan, np.nan, 0, 0
    I, sd, N = sel['I_mean'].values, sel['I_std'].values, sel['N'].values
    keep     = np.abs(I) > I_FLOOR
    if not keep.any():
        return np.nan, np.nan, 0, 0
    I, sd, N = I[keep], sd[keep], N[keep]
    sem      = np.where((N >= 2) & np.isfinite(sd), sd/np.sqrt(N), np.nan)
    ok       = np.isfinite(sem) & (sem > 0)
    if ok.any():
        w = np.zeros_like(I, dtype=float)
        w[ok]    = 1.0/sem[ok]**2
        I_comb   = float(np.sum(w*I) / np.sum(w))
        sig_comb = float(1.0/np.sqrt(np.sum(w)))
    else:
        I_comb   = float(np.mean(I))
        sig_comb = float(np.std(I, ddof=1)/np.sqrt(len(I))) if len(I) > 1 else np.nan
    return I_comb, sig_comb, int(N.sum()), int(len(sel))


per_T_all = pd.concat([per_T_I_per_H(T) for T in TEMPERATURES], ignore_index=True)

# Build per-target ln|I| table with propagated errors, then weighted-fit each regime
traces_c, fits_c = {}, {}
for H_t in H_TARGETS_C:
    rows = []
    for T, g in per_T_all.groupby('T'):
        I, sI, Ntot, nbins = collapse_field_at_T(g, H_t, H_MATCH_TOL)
        if not np.isfinite(I) or abs(I) <= I_FLOOR:
            continue
        lnI     = np.log(abs(I))
        lnI_err = sI/abs(I) if np.isfinite(sI) else np.nan
        rows.append(dict(T=T, I=I, I_err=sI, lnI=lnI, lnI_err=lnI_err,
                         N_total=Ntot, n_bins=nbins))
    if not rows:
        traces_c[H_t] = pd.DataFrame(columns=['T','I','I_err','lnI','lnI_err','N_total','n_bins'])
        fits_c[H_t]   = dict(low=None, high=None)
        continue
    df = pd.DataFrame(rows).sort_values('T').reset_index(drop=True)
    traces_c[H_t] = df
    df_low, df_high = df[df['T'] <= T_LOW_MAX], df[df['T'] >= T_HIGH_MIN]
    fits_c[H_t] = dict(
        low =linfit_weighted(1.0/df_low ['T'].values, df_low ['lnI'].values, df_low ['lnI_err'].values) if len(df_low ) >= 2 else None,
        high=linfit_weighted(1.0/df_high['T'].values, df_high['lnI'].values, df_high['lnI_err'].values) if len(df_high) >= 2 else None)

# ---- Plot: data + errorbars + dashed cubic spline guide-to-the-eye per field
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
cmap = plt.cm.coolwarm
norm = plt.Normalize(vmin=H_TARGETS_C.min(), vmax=H_TARGETS_C.max())

for H_t in H_TARGETS_C:
    df = traces_c[H_t]
    if df.empty:
        continue
    c     = cmap(norm(H_t))
    inv_T = 1.0/df['T'].values
    order = np.argsort(inv_T)
    ax.errorbar(inv_T[order], df['lnI'].values[order],
                yerr=df['lnI_err'].values[order],
                fmt='--o', mec=c, mfc=c, color=c, ecolor=c,
                ms=5, capsize=2, lw=1.0, alpha=0.85)

ax.axvspan(1.0/T_HIGH_MIN, 1.0/T_LOW_MAX, color='0.88', alpha=0.5, zorder=0)
ax.set_xlabel(r'$1/T$ (1/K)')
ax.set_ylabel(rf'$\ln|I (A)|$')
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.85, pad=0.02)
cbar.set_label(r'$|H| \parallel c$ (T)')
fig.tight_layout()
fig.savefig(OUT / 'step1_field_dependent_arrhenius_c_axis.png', dpi=300, bbox_inches='tight')
plt.show()

# ---- Numerical table (cell output, also saved to CSV)
print('Per-field two-regime Arrhenius along the c-axis (folded +H / -H, weighted fit):\n')
print(f'{"|H|(T)":>7s}   {"Ea_low (meV)":>16s}  {"chi2r":>6s}  {"N":>3s}     {"Ea_high (meV)":>16s}  {"chi2r":>6s}  {"N":>3s}')
print('-'*92)
records = []
for H_t in H_TARGETS_C:
    f_low, f_high = fits_c[H_t]['low'], fits_c[H_t]['high']
    def _fmt(f):
        if f is None or not np.isfinite(f['slope']):
            return '         ---   ', np.nan, 0
        Ea     = -f['slope']*k_B*1000
        Ea_err =  f['slope_err']*k_B*1000
        return f'{Ea:6.2f} +/- {Ea_err:5.2f}', f['chi2_red'], f['n']
    low_str, chl, nl = _fmt(f_low)
    high_str, chh, nh = _fmt(f_high)
    print(f'{H_t:7.3f}   {low_str:>16s}  {chl:6.2f}  {nl:3d}     {high_str:>16s}  {chh:6.2f}  {nh:3d}')
    records.append(dict(
        H_T=H_t,
        Ea_low_meV     = -f_low ['slope']    *k_B*1000 if f_low  is not None else np.nan,
        Ea_low_err_meV =  f_low ['slope_err']*k_B*1000 if f_low  is not None else np.nan,
        chi2red_low    =  f_low ['chi2_red']           if f_low  is not None else np.nan,
        N_low          =  f_low ['n']                  if f_low  is not None else 0,
        Ea_high_meV    = -f_high['slope']    *k_B*1000 if f_high is not None else np.nan,
        Ea_high_err_meV=  f_high['slope_err']*k_B*1000 if f_high is not None else np.nan,
        chi2red_high   =  f_high['chi2_red']           if f_high is not None else np.nan,
        N_high         =  f_high['n']                  if f_high is not None else 0,
    ))
EA_FIELD_C = pd.DataFrame(records)
EA_FIELD_C.to_csv(OUT / 'step1_field_dependent_arrhenius_c_axis.csv', index=False)
print('\nSaved table to', OUT / 'step1_field_dependent_arrhenius_c_axis.csv')

# ---- Plot: data + errorbars + dashed cubic spline guide-to-the-eye per field
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
cmap = plt.cm.coolwarm   # blue (low |H|) -> red (high |H|)
norm = plt.Normalize(vmin=H_TARGETS_C.min(), vmax=H_TARGETS_C.max())

for H_t in H_TARGETS_C:
    df = traces_c[H_t]
    if df.empty:
        continue
    c     = cmap(norm(H_t))
    inv_T = 1.0/df['T'].values
    order = np.argsort(inv_T)
    ax.errorbar(inv_T[order], df['lnI'].values[order],
                yerr=df['lnI_err'].values[order],
                fmt='--o', mec=c, mfc=c, color=c, ecolor=c,
                ms=5, capsize=2, lw=1.0, alpha=0.85)

ax.axvspan(1.0/T_HIGH_MIN, 1.0/T_LOW_MAX, color='0.88', alpha=0.5, zorder=0)
ax.set_xlabel(r'$1/T$ (1/K)')
ax.set_ylabel(r'$\ln|I\,(\mathrm{A})|$')
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.85, pad=0.02)
cbar.set_label(r'$|H| \parallel c$ (T)')
fig.tight_layout()
fig.savefig(OUT / 'step1_field_dependent_arrhenius_c_axis.png', dpi=300, bbox_inches='tight')
plt.show()



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 5), dpi=300)

ax = axes[0]
ax.errorbar(FITS['V_probe'], FITS['E_AFM_meV'], yerr=FITS['E_AFM_err_meV'],
            fmt='o-', color=OKABE_ITO_CYCLE[1], ms=5, capsize=2, lw=1.0)
ax.axhline(E_AFM_REF, color=OKABE_ITO_CYCLE[1], lw=0.8, ls=':',
           label=f'V=0.20 V ref = {E_AFM_REF:.2f} meV')
ax.set_xlabel(r'$V_{\rm probe}$ (V)')
ax.set_ylabel(r'$E_a^{\rm AFM}$ (meV)')

ax = axes[1]
ax.errorbar(FITS['V_probe'], FITS['E_FM_meV'], yerr=FITS['E_FM_err_meV'],
            fmt='o-', color=OKABE_ITO_CYCLE[5], ms=5, capsize=2, lw=1.0)
ax.axhline(E_FM_REF, color=OKABE_ITO_CYCLE[5], lw=0.8, ls=':',
           label=f'V=0.20 V ref = {E_FM_REF:.2f} meV')
ax.set_xlabel(r'$V_{\rm probe}$ (V)')
ax.set_ylabel(r'$E_a^{\rm FM}$ (meV)')


ax = axes[2]
ax.errorbar(FITS['V_probe'], FITS['dE_meV'],
            yerr=np.sqrt(FITS['E_AFM_err_meV']**2 + FITS['E_FM_err_meV']**2),
            fmt='o-', color=OKABE_ITO_CYCLE[0], ms=5, capsize=2, lw=1.0)
ax.set_xlabel(r'$V_{\rm probe}$ (V)')
ax.set_ylabel(r'$\Delta E_a$ (meV)')

fig.tight_layout()
fig.savefig(OUT / 'step4_cant_fit_params_vs_Vprobe.png', dpi=300, bbox_inches='tight')
plt.show()

## Save numerical results

All CSVs go to `tmp/arrhenius_variable_Vprobe/figures_story/`. Mirrors what each sibling notebook already saves, kept here for one-stop reproducibility.

In [ ]:
ENDPT_DF.to_csv(OUT / 'step1_endpoints_lnI_c_axis.csv', index=False)
EA_main.to_csv (OUT / 'step2_Ea_vs_H_c_axis_Vprobe_0.20V.csv', index=False)
EA_sweep.to_csv(OUT / 'step4_Ea_vs_H_Vprobe_sweep.csv', index=False)
FITS.to_csv    (OUT / 'step4_cant_fit_vs_Vprobe.csv', index=False)
pd.DataFrame({
    'T_eff_AFM': T_e_A, 'W_AFM': W_A, 'sigW_AFM': sW_A,
    'T_eff_FM':  T_e_F, 'W_FM':  W_F, 'sigW_FM':  sW_F,
}).to_csv(OUT / 'step3_zabrodskii_W_vs_T.csv', index=False)
print('Saved CSVs to', OUT)